# 01 · Data Understanding
**Project:** Explainable Chronic Kidney Disease Prediction Using Classical Machine Learning

> ⚠️ *This project is an educational machine learning demonstration and is not intended for medical diagnosis, clinical decision-making, or patient treatment.*

## 1. Objective
Before any modelling we must know **what the data actually looks like**. In this notebook we load the raw CSV, list its columns, types, missing values and duplicates, discover formatting problems, and write a data dictionary. We do *not* clean anything yet - that is notebook 03.

**Concept - Classification.** Classification means predicting a *category* (here: `ckd` vs `notckd`) from input features (lab tests, symptoms). A binary classifier learns a rule from labelled examples and applies it to new patients.

## 2. Imports

In [1]:
import sys, warnings
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
FIG = Path.cwd().parent / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def savefig(name): plt.savefig(FIG / name, bbox_inches="tight", dpi=150)

## 3. Dataset loading
The loader searches `data/raw/` for a CSV whose *columns* look like the CKD data, instead of trusting a filename.

In [2]:
from src.data_preprocessing import find_dataset, load_raw
from src.feature_engineering import NUMERIC_FEATURES, CATEGORICAL_FEATURES, data_dictionary
path = find_dataset()
print("Detected file:", path.name)
df = load_raw(path)
display(df.head()); display(df.tail())
print("Shape (rows, columns):", df.shape)
print("Columns:", list(df.columns))

Detected file: kidney_disease.csv


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35,7300,4.6,no,no,no,good,no,no,ckd


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
395,395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,49.0,0.5,150.0,4.9,15.7,47,6700,4.9,no,no,no,good,no,no,notckd
396,396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,31.0,1.2,141.0,3.5,16.5,54,7800,6.2,no,no,no,good,no,no,notckd
397,397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,26.0,0.6,137.0,4.4,15.8,49,6600,5.4,no,no,no,good,no,no,notckd
398,398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,50.0,1.0,135.0,4.9,14.2,51,7200,5.9,no,no,no,good,no,no,notckd
399,399,58.0,80.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,131.0,18.0,1.1,141.0,3.5,15.8,53,6800,6.1,no,no,no,good,no,no,notckd


Shape (rows, columns): (400, 26)
Columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


## 4. Data understanding
### 4.1 Types and summary statistics
`df.info()` shows dtypes. **Watch for mismatches**: a column like packed cell volume should be numeric, but pandas will read it as text if even one cell contains a stray character.

In [3]:
df.info()
display(df.describe(include="all").T)

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    str    
 7   pc              335 non-null    str    
 8   pcc             396 non-null    str    
 9   ba              396 non-null    str    
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    str    
 17  wc              295 non-null    str    
 18  r

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,400.0,NaN,NaN,NaN,199.5,115.614301,0.0,99.75,199.5,299.25,399.0
age,391.0,NaN,NaN,NaN,51.483376,17.169714,2.0,42.0,55.0,64.5,90.0
bp,388.0,NaN,NaN,NaN,76.469072,13.683637,50.0,70.0,80.0,80.0,180.0
sg,353.0,NaN,NaN,NaN,1.017408,0.005717,1.005,1.01,1.02,1.02,1.025
al,354.0,NaN,NaN,NaN,1.016949,1.352679,0.0,0.0,0.0,2.0,5.0
su,351.0,NaN,NaN,NaN,0.450142,1.099191,0.0,0.0,0.0,0.0,5.0
rbc,248,2,normal,201,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pc,335,2,normal,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pcc,396,2,notpresent,354,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ba,396,2,notpresent,374,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**What the output means.** `pcv`, `wc` and `rc` (numeric lab values) show up as text (`str`) columns, and `dm`, `cad`, `classification` have more distinct values than they should. Both are red flags - let's look at the raw unique values with `repr()` so invisible characters (tabs, spaces) become visible.

In [4]:
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in text_cols:
    vals = df[c].map(repr).value_counts(dropna=False)
    if c in ("pcv", "wc", "rc"):   # too many numeric-looking values: show only the non-numeric ones
        bad = [v for v in df[c].dropna().unique() if not v.strip().replace(".", "").isdigit()]
        print(f"{c:15s} numeric stored as text; non-numeric entries: {[repr(b) for b in bad]}")
    else:
        print(f"{c:15s} {vals.to_dict()}")

rbc             {"'normal'": 201, 'nan': 152, "'abnormal'": 47}
pc              {"'normal'": 259, "'abnormal'": 76, 'nan': 65}
pcc             {"'notpresent'": 354, "'present'": 42, 'nan': 4}
ba              {"'notpresent'": 374, "'present'": 22, 'nan': 4}
pcv             numeric stored as text; non-numeric entries: ["'\\t?'"]
wc              numeric stored as text; non-numeric entries: ["'\\t?'"]
rc              numeric stored as text; non-numeric entries: ["'\\t?'"]
htn             {"'no'": 251, "'yes'": 147, 'nan': 2}
dm              {"'no'": 258, "'yes'": 134, "'\\tno'": 3, "'\\tyes'": 2, 'nan': 2, "' yes'": 1}
cad             {"'no'": 362, "'yes'": 34, "'\\tno'": 2, 'nan': 2}
appet           {"'good'": 317, "'poor'": 82, 'nan': 1}
pe              {"'no'": 323, "'yes'": 76, 'nan': 1}
ane             {"'no'": 339, "'yes'": 60, 'nan': 1}
classification  {"'ckd'": 248, "'notckd'": 150, "'ckd\\t'": 2}


**Problems found in the raw data**
- Tabs / spaces inside categories: `'\tno'`, `' yes'`, `'\tyes'`, and the target `'ckd\t'`. If untreated, a model would see `'no'` and `'\tno'` as *different* categories.
- `'\t?'` placeholders inside `pcv`, `wc`, `rc` force these numeric columns to be text.
- Missing values are stored as empty cells (NaN) in most columns.
- An `id` column that is only a row number.

### 4.2 Numerical vs categorical columns
We decide types from the actual data + medical meaning (not just dtype). `sg`, `al`, `su` are stored as numbers but are ordinal dipstick grades; we treat them as ordered numeric values.

In [5]:
raw_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != "id"]
print("Numeric in raw file        :", raw_numeric)
print("Numeric after fixing text  :", NUMERIC_FEATURES)
print("Categorical                :", CATEGORICAL_FEATURES)
print("Target column              : classification")
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["id", "classification"]) == set(df.columns)
print(f"\n{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {len(NUMERIC_FEATURES)+len(CATEGORICAL_FEATURES)} predictors (+ id + target = {df.shape[1]} columns)")

Numeric in raw file        : ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo']
Numeric after fixing text  : ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
Categorical                : ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
Target column              : classification

14 numeric + 10 categorical = 24 predictors (+ id + target = 26 columns)


### 4.3 Missing values, duplicates, unique values

In [6]:
miss = df.isna().sum().rename("missing").to_frame()
miss["pct"] = (miss.missing / len(df) * 100).round(1)
display(miss[miss.missing > 0].sort_values("missing", ascending=False).T)
print("Exact duplicate rows (with id)   :", df.duplicated().sum())
print("Exact duplicate rows (id dropped):", df.drop(columns="id").duplicated().sum())
print("\nUnique values per column (raw):")
display(df.nunique().to_frame("n_unique").T)

,rbc,rc,wc,pot,sod,pcv,pc,hemo,su,sg,al,bgr,bu,sc,bp,age,pcc,ba,htn,dm,cad,appet,pe,ane
missing,152.0,130.0,105.0,88.0,87.0,70.0,65.0,52.0,49.0,47.0,46.0,44.0,19.0,17.0,12.0,9.0,4.0,4.0,2.0,2.0,2.0,1.0,1.0,1.0
pct,38.0,32.5,26.2,22.0,21.8,17.5,16.2,13.0,12.2,11.8,11.5,11.0,4.8,4.2,3.0,2.2,1.0,1.0,0.5,0.5,0.5,0.2,0.2,0.2


Exact duplicate rows (with id)   : 0
Exact duplicate rows (id dropped): 0

Unique values per column (raw):


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
n_unique,400,76,10,5,6,6,2,2,2,2,146,118,84,34,40,115,44,92,49,2,5,3,2,2,2,3


### 4.4 Target distribution (raw)
Note the target has **three** labels in the raw file because of one formatting glitch (`'ckd\t'`).

In [7]:
print(df["classification"].map(repr).value_counts())

classification
'ckd'       248
'notckd'    150
'ckd\t'       2
Name: count, dtype: int64


### 4.5 Data dictionary

In [8]:
dd = data_dictionary()
display(dd.style.hide(axis="index"))

column,label,type,unit_or_values,description
age,Age,numeric,years,Patient age
bp,Blood Pressure,numeric,mm/Hg,Blood pressure reading
sg,Specific Gravity,ordinal,1.005-1.025,Urine concentration; low/fixed values suggest reduced kidney concentrating ability
al,Albumin,ordinal,0-5,Urine albumin (protein) dipstick grade
su,Sugar,ordinal,0-5,Urine sugar dipstick grade
rbc,Red Blood Cells (urine),categorical,normal / abnormal,Red blood cells in urine
pc,Pus Cell,categorical,normal / abnormal,Pus cells in urine
pcc,Pus Cell Clumps,categorical,present / notpresent,Clumped pus cells in urine
ba,Bacteria,categorical,present / notpresent,Bacteria in urine
bgr,Blood Glucose Random,numeric,mgs/dl,Random blood glucose


## 10. Conclusions
- 400 rows, 26 columns = `id` + **24 predictors** (14 numeric/ordinal, 10 categorical) + the class label.
- Raw data is *not* analysis-ready: whitespace/tab artifacts, `?` placeholders, numeric columns stored as text, heavy missingness (some columns are missing for a third or more of patients).
- No exact duplicate rows.